# hs-classifier — Example Usage

This notebook walks through the full pipeline:

1. **Setup** — configure logging, load environment
2. **Initialize** — build the FAISS lookup index (one-time)
3. **Classify** — classify product descriptions into HS codes
4. **Split** — create a stratified eval sample using semantic clustering
5. **Evaluate** — compute classification metrics against ground truth

## 1. Setup

In [ ]:
import logging
import warnings

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(name)s | %(message)s")
for name in ("httpx", "faiss.loader", "sentence_transformers", "numba"):
    logging.getLogger(name).setLevel(logging.WARNING)

## 2. Initialize the lookup index

Run once to build the FAISS index from the Atlas DB. Skips automatically if the index already exists.

In [ ]:
from hs_classifier import init_index

init_index()  # pass force=True to rebuild

## 3. Classify a single row

Load the classifier (FAISS index + S-BERT model), then classify a product description.

In [ ]:
from hs_classifier import init_classifier, classify_row

classifier = init_classifier()

In [ ]:
row = {
    "product_description": "frozen shrimp raw peeled",
    "container_description": "1 CONTAINER IC 20 FEET",
}

result = classify_row(row, classifier)

print(f"1st: {result.code_first} — {result.desc_first}")
print(f"2nd: {result.code_second} — {result.desc_second}")
print(f"Reason: {result.reason}")
print(f"Search terms: {result.search_terms}")
print(f"Language: {result.detected_language}")

## 4. Classify from a CSV

Load rows from a CSV file and classify them.

In [ ]:
import polars as pl

df = pl.read_csv("data/raw/ecuador_sample.csv")
print(f"{len(df)} rows, columns: {df.columns}")
df.head(3)

In [ ]:
# classify a single row from the CSV
row = df.row(1, named=True)
result = classify_row(row, classifier)

print(f"Input: {row['product_description']}")
print(f"1st: {result.code_first} — {result.desc_first}")
print(f"2nd: {result.code_second} — {result.desc_second}")
print(f"Reason: {result.reason}")

## 5. Stratified eval sample (splitter)

The splitter embeds product descriptions with S-BERT, reduces dimensions with UMAP, clusters with HDBSCAN, then draws a stratified sample. It only needs the text column — all other columns are carried through.

This is useful for creating representative eval sets from large datasets.

In [ ]:
from sentence_transformers import SentenceTransformer
from hs_classifier.splitter import assign_clusters, stratified_sample, prepare_eval_sample

# load the same embedding model used by the classifier
import os
model = SentenceTransformer(os.environ["EMBEDDING_MODEL"], local_files_only=True)

### Step by step: cluster then sample

In [ ]:
# step 1: assign clusters
clustered = assign_clusters(df, text_col="product_description", model=model, min_cluster_size=5)
clustered.group_by("cluster").agg(pl.len().alias("n")).sort("cluster")

In [ ]:
# step 2: draw a 10% stratified sample
sample = stratified_sample(clustered, sample_frac=0.10)
print(f"Sample: {len(sample)} / {len(clustered)} rows ({len(sample)/len(clustered):.1%})")
sample.head(5)

### Or all-in-one

In [ ]:
sample = prepare_eval_sample(df, text_col="product_description", model=model, sample_frac=0.05)
print(f"Sample: {len(sample)} rows")
sample.head()

### Save the sample

In [ ]:
output_path = "data/raw/ecuador_sample_eval_5pct.csv"
sample.write_csv(output_path)
print(f"Saved to: {output_path}")

## 6. Evaluate predictions (evaluator)

The evaluator compares predicted HS codes against ground truth. It requires a DataFrame with prediction columns (`code_first`, `code_second`) and a truth column.

Here we use the fake test dataset to demonstrate — in practice, you'd run the classifier on your eval sample first, then evaluate.

In [ ]:
from hs_classifier.evaluator import evaluation_report, report_to_markdown

# fake predictions for demo — normally these come from classify_row()
eval_df = pl.DataFrame({
    "code_true":   ["0306", "0306", "0603", "0603", "0803", "1801", "4407", "7108", "7108", "0803"],
    "code_first":  ["0306", "0307", "0603", "0603", "0803", "1801", "4407", "7108", "7113", "1801"],
    "code_second": ["0307", "0306", "0604", "0602", "0804", "1806", "4408", "7113", "7108", "0803"],
})
eval_df

In [ ]:
report = evaluation_report(eval_df, truth_col="code_true")

print(f"Top-1 Accuracy: {report['top1_accuracy']:.1%}")
print(f"Top-K Accuracy: {report['topk_accuracy']:.1%}")
print(f"Chapter Accuracy: {report['chapter_accuracy']:.1%}")

In [ ]:
# per-chapter breakdown
report["per_chapter"]

In [ ]:
# confusion matrix at chapter level
report["confusion_matrix"]

### Markdown report

Render the full evaluation as markdown — useful for saving to a file or pasting into docs.

In [ ]:
from IPython.display import Markdown

md = report_to_markdown(report)
Markdown(md)

In [ ]:
# save the report to a file
with open("data/eval_report.md", "w") as f:
    f.write(md)
print("Saved to: data/eval_report.md")